In [13]:
import sys
import os
from pathlib import Path

# Find the project root (the folder containing pyproject.toml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

sys.path.insert(0, str(ROOT))   # makes `src` and `evals` importable
os.chdir(ROOT)                  # makes "goldens/retriever_goldens.json" resolve correctly

print("Project root:", ROOT)

Project root: c:\Users\gurpr\OneDrive\Desktop\RAG-EVAL


In [ ]:
# eval_retriever.py
import os
from dotenv import load_dotenv

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRecallMetric, ContextualPrecisionMetric
from deepeval.models import GeminiModel

from src.reranker import RerankingRetriever
from evals.harness import load_goldens, summarize_by_metric, print_summary

load_dotenv()   # must run before the judge is created

GOLDEN_PATH = "goldens/retriever_goldens.json"
GEMINI_MODEL = "gemini-2.5-flash"
THRESHOLD = 0.7

# Free Gemini judge, key read from .env (GOOGLE_API_KEY)
import time

class ThrottledGemini(GeminiModel):
    """Gemini judge that stays under the free-tier 5 requests/minute limit."""
    MIN_INTERVAL = 13   # seconds between calls (5/min = one every 12s)
    _last_call = 0.0

    def generate(self, *args, **kwargs):
        for attempt in range(6):
            wait = self.MIN_INTERVAL - (time.time() - ThrottledGemini._last_call)
            if wait > 0:
                time.sleep(wait)
            ThrottledGemini._last_call = time.time()
            try:
                return super().generate(*args, **kwargs)
            except Exception as e:
                if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) and attempt < 5:
                    time.sleep(20)   # quota hit, wait and retry
                    continue
                raise

    async def a_generate(self, *args, **kwargs):
        return self.generate(*args, **kwargs)


judge = ThrottledGemini(
    model=GEMINI_MODEL,
    api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,
)

def run(retriever):
    # 1. LOAD the golden set --- the fixed, human-authored truth
    goldens = load_goldens(GOLDEN_PATH)

    # 2. RUN THE INJECTED RETRIEVER on each question to fill retrieval_context,
    #    then build one test case per golden.
    test_cases = []
    for g in goldens:
        retrieved = retriever.invoke(g["query"])
        retrieval_context = [doc.page_content for doc in retrieved]

        test_cases.append(
            LLMTestCase(
                input=g["query"],
                expected_output=g["ideal_answer"],
                retrieval_context=retrieval_context,
                actual_output="(generator not evaluated in this run)",
            )
        )

    # 3. THE METRICS --- recall (did we miss?) and precision (ranked well?)
    #    async_mode=False keeps requests sequential, safest with free-tier rate limits.
    metrics = [
        ContextualRecallMetric(
            threshold=THRESHOLD, model=judge, include_reason=True, async_mode=False
        ),
        ContextualPrecisionMetric(
            threshold=THRESHOLD, model=judge, include_reason=True, async_mode=False
        ),
    ]

    # 4. EVALUATE --- every metric on every case, printed report.
    #    hyperparameters travel with the run so the report is tagged with the config.
    result = evaluate(
        test_cases=test_cases,
        metrics=metrics,
        hyperparameters={
            "retriever": "reranker",
            "embedding_model": "all-MiniLM-L6-v2",
            "chunk_size": 1000,
            "chunk_overlap": 150,
            "top_k": 5,                  # reranker's top_k
            "judge_model": GEMINI_MODEL,
            "golden_set": GOLDEN_PATH,
        },
    )
    return summarize_by_metric(result)


def run_local():
    """Standalone convenience: build the retriever, then run."""
    return run(RerankingRetriever())


if __name__ == "__main__":
    print_summary("retriever", run_local())

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4258.75it/s]


✨ You're running DeepEval's latest Contextual Recall Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

c:\Users\gurpr\OneDrive\Desktop\RAG-EVAL\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')